In [ ]:
pip install statsmodels

In [ ]:
%%writefile arimax_core.py
import numpy as np
from scipy.optimize import minimize
from scipy import stats


def _difference(a, d):
    """d-th order differencing along axis 0. Works for 1D or 2D arrays."""
    out = a
    for _ in range(d):
        out = np.diff(out, axis=0)
    return out


class ARIMAXResult:
    def __init__(self, order, beta, phi, theta, eps, resid_reg, sigma2,
                 aic, k_params, n_obs, y_hist, x_design_cols, d, has_const,
                 last_x_row, x_mean=None, x_std=None):
        self.order = order
        self.beta = beta
        self.phi = phi
        self.theta = theta
        self.eps = eps
        self.resid_reg = resid_reg
        self.sigma2 = sigma2
        self.aic = aic
        self.k_params = k_params
        self.n_obs = n_obs
        self.y_hist = y_hist          # original (log-scale) endog used in fit
        self.d = d
        self.has_const = has_const
        self.last_x_row = last_x_row


def fit_arimax(y, X, order, maxiter=200):
    """
    y : 1D np.array, log-scale endogenous series (already aligned/sorted)
    X : 2D np.array (n, k_exog), exogenous regressors, same length as y
    order : (p, d, q)
    Returns ARIMAXResult or None on failure.
    """
    p, d, q = order
    n = len(y)
    if n - d - max(p, q) < max(8, p + q + 3):
        return None  # not enough data to identify the model

    has_const = (d == 0)
    if has_const:
        Xc = np.column_stack([np.ones(n), X])
    else:
        Xc = X.copy()

    yd = _difference(y, d)
    Xd = _difference(Xc, d)
    m = len(yd)

    k = Xd.shape[1]
    n_params = k + p + q

    # Initial guess: OLS beta, zero AR/MA coefficients
    beta0, *_ = np.linalg.lstsq(Xd, yd, rcond=None)
    x0 = np.concatenate([beta0, np.zeros(p), np.zeros(q)])

    burn = max(p, q)

    def unpack(params):
        beta = params[:k]
        phi = params[k:k + p]
        theta = params[k + p:k + p + q]
        return beta, phi, theta

    def sse(params):
        beta, phi, theta = unpack(params)
        w = yd - Xd @ beta
        eps = np.zeros(m)
        for t in range(m):
            ar = 0.0
            for i in range(p):
                if t - i - 1 >= 0:
                    ar += phi[i] * w[t - i - 1]
            ma = 0.0
            for j in range(q):
                if t - j - 1 >= 0:
                    ma += theta[j] * eps[t - j - 1]
            eps[t] = w[t] - ar - ma
        usable = eps[burn:]
        return np.sum(usable ** 2)

    bounds = [(-1e6, 1e6)] * k + [(-1.98, 1.98)] * p + [(-1.98, 1.98)] * q

    try:
        res = minimize(sse, x0, method='L-BFGS-B', bounds=bounds,
                        options={'maxiter': maxiter, 'ftol': 1e-10})
    except Exception:
        return None

    if not np.all(np.isfinite(res.x)):
        return None

    beta, phi, theta = unpack(res.x)
    w = yd - Xd @ beta
    eps = np.zeros(m)
    for t in range(m):
        ar = sum(phi[i] * w[t - i - 1] for i in range(p) if t - i - 1 >= 0)
        ma = sum(theta[j] * eps[t - j - 1] for j in range(q) if t - j - 1 >= 0)
        eps[t] = w[t] - ar - ma

    usable = eps[burn:]
    n_obs = len(usable)
    ssev = np.sum(usable ** 2)
    if n_obs <= 0 or ssev <= 0:
        return None
    sigma2 = ssev / n_obs
    k_params = n_params + 1  # +1 for sigma^2
    aic = n_obs * np.log(sigma2) + 2 * k_params

    last_x_row = Xc[-1, :]

    return ARIMAXResult(order, beta, phi, theta, eps, w, sigma2, aic,
                         k_params, n_obs, y, Xd.shape[1], d, has_const,
                         last_x_row)


def forecast_arimax(fit_result, x_new_row, y_last, x_prev_row):
    p, d, q = fit_result.order
    beta, phi, theta = fit_result.beta, fit_result.phi, fit_result.theta
    w_hist = fit_result.resid_reg
    eps_hist = fit_result.eps

    if fit_result.has_const:
        x_new = np.concatenate([[1.0], x_new_row])
        x_prev = np.concatenate([[1.0], x_prev_row])
    else:
        x_new = x_new_row
        x_prev = x_prev_row

    if d == 1:
        x_design = x_new - x_prev
    else:
        x_design = x_new

    reg_part = x_design @ beta

    ar_term = 0.0
    for i in range(p):
        idx = len(w_hist) - 1 - i
        if idx >= 0:
            ar_term += phi[i] * w_hist[idx]
    ma_term = 0.0
    for j in range(q):
        idx = len(eps_hist) - 1 - j
        if idx >= 0:
            ma_term += theta[j] * eps_hist[idx]

    w_forecast = reg_part + ar_term + ma_term  # predicted Delta^d eta_t

    if d == 1:
        y_forecast = y_last + w_forecast
    else:
        y_forecast = w_forecast

    se = np.sqrt(fit_result.sigma2)
    return y_forecast, se


def ljung_box(resid, lags=(6, 12, 24)):
    """Manual Ljung-Box test (no statsmodels dependency)."""
    n = len(resid)
    resid = np.asarray(resid) - np.mean(resid)
    acf_full = np.correlate(resid, resid, mode='full')
    mid = len(acf_full) // 2
    c0 = acf_full[mid]
    rows = []
    for h in lags:
        if h >= n:
            rows.append((h, np.nan, np.nan))
            continue
        stat = 0.0
        for k in range(1, h + 1):
            ck = acf_full[mid + k]
            rk = ck / c0
            stat += (rk ** 2) / (n - k)
        stat *= n * (n + 2)
        pval = stats.chi2.sf(stat, df=h)
        rows.append((h, stat, pval))
    return rows

Overwriting arimax_core.py


In [ ]:
import warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

from arimax_core import fit_arimax, forecast_arimax, ljung_box

# CONFIG
CSV_PATH    = "/content/Dengue_Climate_Bangladesh.csv"
OUTPUT_PNG = "arimax_prediction.png"
DIAG_PNG = "arimax_residual_diagnostics.png"
RESULTS_CSV = "arimax_2025_predictions.csv"

# 1. DATA LOADING & PREPROCESSING
df = pd.read_csv(CSV_PATH)
df['DATE'] = pd.to_datetime(df['YEAR'].astype(str) + '-' + df['MONTH'].astype(str) + '-01')
df = df.sort_values('DATE').reset_index(drop=True)
df.set_index('DATE', inplace=True)
df.index.freq = 'MS'

# 2. FEATURE ENGINEERING — DENGUE lags + CLIMATE lags
df['TEMP'] = (df['MIN'] + df['MAX']) / 2

# --- Dengue's own history (autoregressive-style features) ---
df['DENGUE_lag_1']  = df['DENGUE'].shift(1)    # previous month
df['DENGUE_lag_2']  = df['DENGUE'].shift(2)    # two months ago
df['DENGUE_lag_12'] = df['DENGUE'].shift(12)   # same month, one year ago (seasonal)

# --- Climate history (lag 1 = previous month, lag 2 = two months ago) ---
df['RAINFALL_lag_1'] = df['RAINFALL'].shift(1)
df['RAINFALL_lag_2'] = df['RAINFALL'].shift(2)
df['HUMIDITY_lag_1'] = df['HUMIDITY'].shift(1)
df['HUMIDITY_lag_2'] = df['HUMIDITY'].shift(2)
df['TEMP_lag_1']     = df['TEMP'].shift(1)
df['TEMP_lag_2']     = df['TEMP'].shift(2)

df_clean = df.dropna().copy()
df_clean.index.freq = 'MS'

# 3. SYSTEM MATRICES & TRAIN-TEST SPLITTING
endog = df_clean['DENGUE']
endog_log = np.log1p(endog)

EXOG_COLS = [
    'DENGUE_lag_1', 'DENGUE_lag_2', 'DENGUE_lag_12',
    'RAINFALL_lag_1', 'RAINFALL_lag_2',
    'HUMIDITY_lag_1', 'HUMIDITY_lag_2',
    'TEMP_lag_1', 'TEMP_lag_2',
]
exog = df_clean[EXOG_COLS]

TRAIN_END  = '2024-12-01'
TEST_START = '2025-01-01'
TEST_END   = '2025-10-01'

train_endog = endog_log[:TRAIN_END]
train_exog  = exog[:TRAIN_END]
pred_dates  = exog[TEST_START:TEST_END].index

print(f"Training window : {train_endog.index[0].strftime('%b-%Y')} -> {train_endog.index[-1].strftime('%b-%Y')}  (n={len(train_endog)})")
print(f"Test window      : {pred_dates[0].strftime('%b-%Y')} -> {pred_dates[-1].strftime('%b-%Y')}  (n={len(pred_dates)})")
print(f"Exogenous features ({len(EXOG_COLS)}): {EXOG_COLS}")

# Sanity check: exactly what feeds the Jan-2025 forecast
jan25 = exog.loc['2025-01-01']
print("\nFeatures driving the Jan-2025 forecast (all known BEFORE Jan-2025):")
print(f"  DENGUE_lag_1   (Dec-2024 dengue)   = {jan25['DENGUE_lag_1']:.0f}")
print(f"  DENGUE_lag_2   (Nov-2024 dengue)   = {jan25['DENGUE_lag_2']:.0f}")
print(f"  DENGUE_lag_12  (Jan-2024 dengue)   = {jan25['DENGUE_lag_12']:.0f}")
print(f"  RAINFALL_lag_1 (Dec-2024 rainfall) = {jan25['RAINFALL_lag_1']:.1f}")
print(f"  RAINFALL_lag_2 (Nov-2024 rainfall) = {jan25['RAINFALL_lag_2']:.1f}")
print(f"  HUMIDITY_lag_1 (Dec-2024 humidity) = {jan25['HUMIDITY_lag_1']:.1f}")
print(f"  HUMIDITY_lag_2 (Nov-2024 humidity) = {jan25['HUMIDITY_lag_2']:.1f}")
print(f"  TEMP_lag_1     (Dec-2024 temp)     = {jan25['TEMP_lag_1']:.1f}")
print(f"  TEMP_lag_2     (Nov-2024 temp)     = {jan25['TEMP_lag_2']:.1f}")


# 4. AUTOMATED HYPERPARAMETER AIC GRID SEARCH — NO SEASONAL COMPONENT
def optimize_arimax(y, X):
    best_aic = float('inf')
    best_order = None
    best_res = None
    print("Executing Optimization Grid Search Across (p,d,q) Space [p:0-13, d:0-1, q:0-2]...")
    for p in range(0, 14):
        for d in range(0, 2):
            for q in range(0, 3):
                res = fit_arimax(y, X, order=(p, d, q))
                if res is None:
                    continue
                if res.aic < best_aic:
                    best_aic = res.aic
                    best_order = (p, d, q)
                    best_res = res
    print(f"Optimal ARIMAX Order Selected: {best_order} | Lowest AIC: {best_aic:.2f}")
    return best_order, best_res

y_arr = train_endog.values
X_arr = train_exog.values
optimal_order, _ = optimize_arimax(y_arr, X_arr)

# 5. MODEL TRAINING & RESIDUAL DIAGNOSTICS
results = fit_arimax(y_arr, X_arr, order=optimal_order, maxiter=500)

lb_rows = ljung_box(results.eps, lags=(6, 12, 24))
lb_df = pd.DataFrame(lb_rows, columns=['lag', 'lb_stat', 'p_value'])
print("\nLjung-Box test on residuals (checks for leftover autocorrelation):")
print(lb_df.to_string(index=False))

# Residual diagnostics figure (ACF, histogram, Q-Q, residuals-over-time)
from scipy import stats as sp_stats

resid = results.eps
BG, PANEL, GRID = '#0d1117', '#161b22', '#21262d'
BORDER, WHITE, MUTED = '#30363d', '#e6edf3', '#8b949e'
BLUE, ORANGE, GREEN, YELLOW = '#58a6ff', '#f78166', '#3fb950', '#d29922'

fig_diag, axd = plt.subplots(2, 2, figsize=(12, 8), dpi=150)
fig_diag.patch.set_facecolor(BG)

def _style(ax, title):
    ax.set_facecolor(PANEL)
    ax.set_title(title, color=WHITE, fontsize=10, fontweight='bold')
    ax.tick_params(colors=MUTED, labelsize=8)
    ax.spines[['top', 'right']].set_visible(False)
    ax.spines[['bottom', 'left']].set_color(BORDER)
    ax.grid(True, color=GRID, linestyle='--', linewidth=0.6, alpha=0.9)

axd[0, 0].plot(resid, color=BLUE, lw=1.1)
axd[0, 0].axhline(0, color=MUTED, lw=0.8, ls='--')
_style(axd[0, 0], 'Standardized Residuals Over Time')

axd[0, 1].hist(resid, bins=20, color=ORANGE, alpha=0.85, density=True)
xs = np.linspace(resid.min(), resid.max(), 200)
axd[0, 1].plot(xs, sp_stats.norm.pdf(xs, resid.mean(), resid.std()), color=WHITE, lw=1.5, label='N(0,σ²)')
_style(axd[0, 1], 'Residual Histogram vs Normal')
axd[0, 1].legend(fontsize=7, facecolor=PANEL, labelcolor=WHITE, edgecolor=BORDER)

sp_stats.probplot(resid, dist='norm', plot=axd[1, 0])
axd[1, 0].get_lines()[0].set_markerfacecolor(BLUE)
axd[1, 0].get_lines()[0].set_markeredgecolor(BLUE)
axd[1, 0].get_lines()[1].set_color(ORANGE)
_style(axd[1, 0], 'Normal Q-Q Plot')

n_r = len(resid)
maxlag = min(24, n_r // 2)
acf_vals = [1.0]
rc = resid - resid.mean()
denom = np.sum(rc ** 2)
for k in range(1, maxlag + 1):
    acf_vals.append(np.sum(rc[:-k] * rc[k:]) / denom)
axd[1, 1].bar(range(len(acf_vals)), acf_vals, color=GREEN, alpha=0.85)
ci_bound = 1.96 / np.sqrt(n_r)
axd[1, 1].axhline(ci_bound, color=YELLOW, lw=1, ls='--')
axd[1, 1].axhline(-ci_bound, color=YELLOW, lw=1, ls='--')
_style(axd[1, 1], 'ACF of Residuals (95% bounds)')

fig_diag.suptitle(f'ARIMAX{optimal_order} Residual Diagnostics — Ljung-Box p(lag12)={lb_df.loc[lb_df.lag==12,"p_value"].values[0]:.3f}',
                   color=WHITE, fontsize=12, fontweight='bold')
fig_diag.tight_layout(rect=[0, 0, 1, 0.95])
fig_diag.savefig(DIAG_PNG, bbox_inches='tight', facecolor=BG, dpi=150)
print(f"Residual diagnostic graphics saved -> {DIAG_PNG}")

# 6. WALK-FORWARD FORECASTING & ACCURACY VERIFICATION (Jan-Oct 2025)
y_pred_list, lb_list, ub_list = [], [], []

print("\nRunning walk-forward (expanding window) 1-step-ahead forecasts...")
for cutoff_date in pred_dates:
    cur_train_endog = endog_log[:cutoff_date].iloc[:-1]
    cur_train_exog  = exog.loc[cur_train_endog.index]
    step_exog_row   = exog.loc[cutoff_date].values
    prev_exog_row   = exog.loc[cur_train_endog.index[-1]].values
    y_last          = cur_train_endog.values[-1]

    step_res = fit_arimax(cur_train_endog.values, cur_train_exog.values,
                           order=optimal_order, maxiter=500)
    if step_res is None:
        # fall back to previous order search result if the fit fails
        step_res = results

    fc_log, se_log = forecast_arimax(step_res, step_exog_row, y_last, prev_exog_row)

    point = np.clip(np.expm1(fc_log), 0, None)
    lo = np.clip(np.expm1(fc_log - 1.96 * se_log), 0, None)
    hi = np.expm1(fc_log + 1.96 * se_log)

    y_pred_list.append(point)
    lb_list.append(lo)
    ub_list.append(hi)
    print(f"  {cutoff_date.strftime('%b-%Y')} forecast complete")

y_true = endog[TEST_START:TEST_END].values.astype(float)
y_pred = np.array(y_pred_list)
lb_arr = np.array(lb_list)
ub_arr = np.array(ub_list)

mae  = mean_absolute_error(y_true, y_pred)
rmse = np.sqrt(mean_squared_error(y_true, y_pred))
r2   = r2_score(y_true, y_pred)

abs_pct_err  = np.where(y_true == 0, np.nan, np.abs(y_true - y_pred) / y_true * 100)
pct_accuracy = 100 - abs_pct_err
mape = np.nanmean(abs_pct_err)
overall_accuracy_pct = 100 - mape

print("\n=== 2025 (JAN-OCT) ARIMAX WALK-FORWARD PREDICTION PERFORMANCE ===")
print(f"  Order : ARIMAX{optimal_order} (no seasonal component)")
print(f"  MAE  : {mae:>10,.0f} cases")
print(f"  RMSE : {rmse:>10,.0f} cases")
print(f"  R2   : {r2:>10.4f}")
print(f"  MAPE : {mape:>10.2f} %")
print(f"  Overall Accuracy (100-MAPE) : {overall_accuracy_pct:>6.2f} %")

results_table = pd.DataFrame({
    'Month': [d.strftime('%b-%Y') for d in pred_dates],
    'Observed': y_true.astype(int),
    'Predicted': np.round(y_pred).astype(int),
    'Lower_95CI': np.round(lb_arr).astype(int),
    'Upper_95CI': np.round(ub_arr).astype(int),
    'Abs_Error': np.round(np.abs(y_true - y_pred)).astype(int),
    'Pct_Error_%': np.round(abs_pct_err, 2),
    'Pct_Accuracy_%': np.round(pct_accuracy, 2),
})
results_table.to_csv(RESULTS_CSV, index=False)
print(f"\nPer-month results saved -> {RESULTS_CSV}")
print(results_table.to_string(index=False))

# 7. VISUAL DASHBOARD GENERATION
def style_ax(ax, title, ylabel='Dengue Cases'):
    ax.set_facecolor(PANEL)
    ax.set_title(title, color=WHITE, fontsize=11, fontweight='bold', pad=8)
    ax.tick_params(colors=MUTED, labelsize=8)
    ax.spines[['top', 'right']].set_visible(False)
    ax.spines[['bottom', 'left']].set_color(BORDER)
    ax.grid(True, color=GRID, linestyle='--', linewidth=0.6, alpha=0.9)
    ax.set_ylabel(ylabel, color=MUTED, fontsize=9)

fig = plt.figure(figsize=(15, 14), dpi=150)
fig.patch.set_facecolor(BG)
gs  = fig.add_gridspec(4, 2, hspace=0.55, wspace=0.32)
ax1 = fig.add_subplot(gs[0, :])
ax2 = fig.add_subplot(gs[1, :])
ax3 = fig.add_subplot(gs[2, 0])
ax4 = fig.add_subplot(gs[2, 1])
ax5 = fig.add_subplot(gs[3, :])

style_ax(ax1, '▸ Walk-Forward System View: Dengue Timeline Verification (2008 – 2025)')
ax1.plot(train_endog.index, np.expm1(train_endog.values), color=MUTED, lw=1.3, alpha=0.55, label='Historical Records')
ax1.plot(pred_dates, y_true, color=BLUE, lw=2, marker='o', ms=4, label='Observed 2025')
ax1.plot(pred_dates, y_pred, color=ORANGE, lw=2, ls='--', marker='x', ms=5, label='ARIMAX Forecast')
ax1.fill_between(pred_dates, lb_arr, ub_arr, color=ORANGE, alpha=0.15, label='95% CI Boundary')
ax1.legend(fontsize=8.5, facecolor=PANEL, labelcolor=WHITE, edgecolor=BORDER, loc='upper left', ncol=3)

style_ax(ax2, '▸ 2025 (Jan–Oct) Walk-Forward Accuracy Resolution')
ax2.plot(pred_dates, y_true, color=BLUE, lw=2.3, marker='o', ms=6, label='Observed Cases (DGHS)', zorder=4)
ax2.plot(pred_dates, y_pred, color=ORANGE, lw=2.3, ls='--', marker='x', ms=7, label='ARIMAX Prediction', zorder=4)
ax2.fill_between(pred_dates, lb_arr, ub_arr, color=ORANGE, alpha=0.18, label='95% Predictive Envelope')

obs_pk, pred_pk = np.argmax(y_true), np.argmax(y_pred)
ax2.annotate(f'Observed Peak\n{int(y_true[obs_pk]):,}', xy=(pred_dates[obs_pk], y_true[obs_pk]),
             xytext=(pred_dates[max(obs_pk-2,0)], y_true[obs_pk]*0.80), color=YELLOW, fontsize=9, arrowprops=dict(arrowstyle='->', color=YELLOW, lw=1.2))
ax2.annotate(f'Predicted Peak\n{int(y_pred[pred_pk]):,}', xy=(pred_dates[pred_pk], y_pred[pred_pk]),
             xytext=(pred_dates[max(pred_pk-2,0)], y_pred[pred_pk]*1.15), color=ORANGE, fontsize=9, arrowprops=dict(arrowstyle='->', color=ORANGE, lw=1.2))

ax2.text(0.99, 0.04, f'MAE: {mae:,.0f} | RMSE: {rmse:,.0f} | R2: {r2:.4f} | Accuracy: {overall_accuracy_pct:.1f}%',
         transform=ax2.transAxes, ha='right', va='bottom', fontsize=9.5, color=WHITE,
         bbox=dict(boxstyle='round,pad=0.5', facecolor=BG, edgecolor=BORDER, alpha=0.92))
ax2.xaxis.set_major_formatter(mdates.DateFormatter('%b %Y'))
ax2.xaxis.set_major_locator(mdates.MonthLocator(interval=1))
plt.setp(ax2.xaxis.get_majorticklabels(), rotation=30, ha='right')
ax2.legend(fontsize=8.5, facecolor=PANEL, labelcolor=WHITE, edgecolor=BORDER, loc='upper left')

style_ax(ax3, '▸ Monthly Convergence Distribution')
months_lbl = [d.strftime('%b') for d in pred_dates]
x, w = np.arange(len(pred_dates)), 0.38
ax3.bar(x - w/2, y_true, w, color=BLUE, alpha=0.85, label='Observed', zorder=3)
ax3.bar(x + w/2, y_pred, w, color=ORANGE, alpha=0.85, label='Predicted', zorder=3)
ax3.set_xticks(x)
ax3.set_xticklabels(months_lbl, fontsize=8.5, color=MUTED)
ax3.legend(fontsize=8.5, facecolor=PANEL, labelcolor=WHITE, edgecolor=BORDER)

style_ax(ax4, '▸ Predictive Error Volatility Analysis', ylabel='Residual Cases')
residuals = y_true - y_pred
res_colors = [GREEN if r >= 0 else ORANGE for r in residuals]
ax4.bar(range(len(pred_dates)), residuals, color=res_colors, alpha=0.85, zorder=3)
ax4.axhline(0, color=WHITE, lw=0.9, ls='--', alpha=0.5)
ax4.set_xticks(range(len(pred_dates)))
ax4.set_xticklabels(months_lbl, fontsize=8.5, color=MUTED)

style_ax(ax5, '▸ Per-Month Prediction Accuracy (%)', ylabel='Accuracy (%)')
bar_colors = [GREEN if a >= 80 else (YELLOW if a >= 60 else ORANGE) for a in np.nan_to_num(pct_accuracy, nan=0)]
ax5.bar(range(len(pred_dates)), np.nan_to_num(pct_accuracy, nan=0), color=bar_colors, alpha=0.9, zorder=3)
ax5.axhline(100, color=WHITE, lw=0.9, ls='--', alpha=0.4)
ax5.set_xticks(range(len(pred_dates)))
ax5.set_xticklabels(months_lbl, fontsize=8.5, color=MUTED)
ax5.set_ylim(min(0, np.nanmin(pct_accuracy) - 5), 105)
for i, a in enumerate(pct_accuracy):
    label = f"{a:.0f}%" if not np.isnan(a) else "N/A"
    ax5.text(i, (0 if np.isnan(a) else a) + (2 if (not np.isnan(a) and a >= 0) else -8), label,
              ha='center', fontsize=8, color=WHITE)

fig.suptitle(f'Walk-Forward ARIMAX Early Warning Matrix — Bangladesh 2025 (Jan–Oct)\nSelected Order: ARIMAX{optimal_order} (no seasonal component) | Exog: {len(EXOG_COLS)} (Dengue+Climate lags) | Accuracy: {overall_accuracy_pct:.1f}%',
             color=WHITE, fontsize=12, fontweight='bold', y=0.999)

plt.savefig(OUTPUT_PNG, bbox_inches='tight', facecolor=BG, dpi=150)
print(f"Publication-ready verification matrix saved -> {OUTPUT_PNG}")

# Save the Ljung-Box table diagnostics section
lb_df.to_csv("arimax_ljungbox.csv", index=False)
print("Ljung-Box table saved -> arimax_ljungbox.csv")


Training window : Jan-2009 -> Dec-2024  (n=192)
Test window      : Jan-2025 -> Oct-2025  (n=10)
Exogenous features (9): ['DENGUE_lag_1', 'DENGUE_lag_2', 'DENGUE_lag_12', 'RAINFALL_lag_1', 'RAINFALL_lag_2', 'HUMIDITY_lag_1', 'HUMIDITY_lag_2', 'TEMP_lag_1', 'TEMP_lag_2']

Features driving the Jan-2025 forecast (all known BEFORE Jan-2025):
  DENGUE_lag_1   (Dec-2024 dengue)   = 9745
  DENGUE_lag_2   (Nov-2024 dengue)   = 29652
  DENGUE_lag_12  (Jan-2024 dengue)   = 1055
  RAINFALL_lag_1 (Dec-2024 rainfall) = 15.2
  RAINFALL_lag_2 (Nov-2024 rainfall) = 55.2
  HUMIDITY_lag_1 (Dec-2024 humidity) = 75.3
  HUMIDITY_lag_2 (Nov-2024 humidity) = 77.6
  TEMP_lag_1     (Dec-2024 temp)     = 20.6
  TEMP_lag_2     (Nov-2024 temp)     = 24.4
Executing Optimization Grid Search Across (p,d,q) Space [p:0-13, d:0-1, q:0-2]...
Optimal ARIMAX Order Selected: (13, 1, 0) | Lowest AIC: -40.66

Ljung-Box test on residuals (checks for leftover autocorrelation):
 lag   lb_stat  p_value
   6  7.371108 0.287886
  1